# Stockky — Decay Profile Calibration (real data pull)

This notebook connects **read-only** to your Oracle Autonomous DB, runs the same
calibration analysis as `scripts/calibrate_decay_profiles.py`, and exports a
small **aggregated summary only** (no raw prices, no symbol-level rows, no
credentials) that you can hand back to Claude to get real, data-backed
`decay.py` recommendations instead of the original estimates.

## Security note, please read first
This notebook asks for your Oracle DB credentials and wallet file to connect.
Colab runs on Google's cloud — treat that the same as pasting a password into
any other cloud tool.
- Credentials are entered via a masked prompt (`getpass`), never typed into a
  visible cell, never saved to disk, never included in any export.
- Nothing is written to the DB — every query below is a `SELECT`.
- **If you'd rather not do this at all**, the safer alternative is to just run
  `python scripts/calibrate_decay_profiles.py --mode REAL --days 30` directly
  on your Oracle VM (where the credentials already live safely in `.env`) and
  paste me the printed output — that never leaves your server. This notebook
  exists for convenience / charts, not because it's required.


## 1. Install the Oracle driver (thin mode — no Instant Client needed)

In [ ]:
!pip install -q oracledb pandas

## 2. Upload your Oracle wallet
Upload the wallet **zip file** you already have for this project (the same one
your `docker-compose.yml` mounts at `/oracle_wallet`). It'll be extracted into
a temp folder for this session only and discarded when the Colab runtime
resets.

In [ ]:
import zipfile, os
from google.colab import files

print("Select your Oracle wallet .zip file:")
uploaded = files.upload()
wallet_zip = list(uploaded.keys())[0]

WALLET_DIR = "/content/oracle_wallet"
os.makedirs(WALLET_DIR, exist_ok=True)
with zipfile.ZipFile(wallet_zip, "r") as z:
    z.extractall(WALLET_DIR)

print("Wallet extracted to:", WALLET_DIR)
print("Contents:", os.listdir(WALLET_DIR))


## 3. Enter connection details (masked, not saved)
Same values as `ORACLE_USER` / `ORACLE_PASSWORD` / `ORACLE_DSN` /
`ORACLE_WALLET_PASSWORD` in your `.env`. `ORACLE_DSN` is the TNS alias from
your wallet's `tnsnames.ora` (e.g. `stockkydb_high`).

In [ ]:
import getpass

ORACLE_USER = input("ORACLE_USER (e.g. ADMIN): ").strip()
ORACLE_PASSWORD = getpass.getpass("ORACLE_PASSWORD: ")
ORACLE_DSN = input("ORACLE_DSN (TNS alias, e.g. stockkydb_high): ").strip()
ORACLE_WALLET_PASSWORD = getpass.getpass("ORACLE_WALLET_PASSWORD (leave blank if none): ")


## 4. Connect (read-only session)

In [ ]:
import oracledb

oracledb.init_oracle_client() if False else None  # thin mode — no init needed, kept explicit for clarity

connection = oracledb.connect(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    config_dir=WALLET_DIR,
    wallet_location=WALLET_DIR,
    wallet_password=ORACLE_WALLET_PASSWORD or None,
)
connection.begin()  # no writes will happen, but keeps this session isolated
print("Connected. This session will only ever run SELECT statements below.")


## 5. Pull settings
Adjust the lookback window / mode if you want. `MIN_SAMPLES` matches the
threshold in `calibrate_decay_profiles.py` — below this, a catalyst type is
reported as "not enough data yet" rather than a number that's mostly noise.

In [ ]:
MODE = "REAL"      # "REAL", "DEMO", or "ALL"
LOOKBACK_DAYS = 30
MIN_SAMPLES = 8


## 6. Run the calibration query per catalyst type

In [ ]:
import pandas as pd

mode_filter = "" if MODE == "ALL" else f"AND w.mode = '{MODE}'"

# trade_watchlist: every catalyst detected, incl. status (active/entered/missed/expired)
watchlist_sql = f"""
    SELECT w.catalyst_type, w.status, w.horizon_class, w.entry_band_pct, w.id AS watchlist_id
    FROM trade_watchlist w
    WHERE w.catalyst_ts >= SYSTIMESTAMP - INTERVAL '{LOOKBACK_DAYS}' DAY
    {mode_filter}
"""
watchlist_df = pd.read_sql(watchlist_sql, connection)
print(f"Pulled {len(watchlist_df)} watchlist rows across {watchlist_df['catalyst_type'].nunique()} catalyst types")
watchlist_df.head()


In [ ]:
# trade_positions closed positions that originated from a watchlist entry,
# joined to their exit decisions to compute time_stop_rate.
positions_sql = f"""
    SELECT
        p.id AS position_id,
        p.watchlist_entry_id,
        p.avg_entry_price,
        p.qty_open,
        p.realized_pnl,
        p.opened_at,
        p.closed_at
    FROM trade_positions p
    WHERE p.status = 'CLOSED'
      AND p.watchlist_entry_id IS NOT NULL
      AND p.closed_at >= SYSTIMESTAMP - INTERVAL '{LOOKBACK_DAYS}' DAY
      {mode_filter.replace('w.mode', 'p.mode')}
"""
positions_df = pd.read_sql(positions_sql, connection)
print(f"Pulled {len(positions_df)} closed positions with a known watchlist origin")

if len(positions_df):
    exit_sql = f"""
        SELECT position_id, action, reasoning
        FROM trade_exit_decisions
        WHERE action = 'FULL_EXIT'
          AND position_id IN ({','.join(str(i) for i in positions_df['position_id'].tolist())})
    """
    exits_df = pd.read_sql(exit_sql, connection)
else:
    exits_df = pd.DataFrame(columns=["position_id", "action", "reasoning"])


## 7. Compute the calibration summary (aggregated only — this is what gets exported)

In [ ]:
import numpy as np

positions_df["held_days"] = (
    pd.to_datetime(positions_df["closed_at"]) - pd.to_datetime(positions_df["opened_at"])
).dt.total_seconds() / 86400
positions_df["pnl_pct"] = (
    positions_df["realized_pnl"] / (positions_df["avg_entry_price"] * positions_df["qty_open"].clip(lower=1)) * 100
)

merged = watchlist_df.merge(
    positions_df, left_on="watchlist_id", right_on="watchlist_entry_id", how="left"
)
time_stop_ids = set(
    exits_df.loc[exits_df["reasoning"].str.contains("time", case=False, na=False) &
                 exits_df["reasoning"].str.contains("stop", case=False, na=False), "position_id"]
)

rows = []
for ctype, grp in merged.groupby("catalyst_type"):
    entered = (grp["status"] == "entered").sum()
    missed = (grp["status"] == "missed").sum()
    total_decided = entered + missed
    missed_rate = missed / total_decided if total_decided else None

    closed = grp.dropna(subset=["position_id"])
    n_closed = len(closed)
    median_days = closed["held_days"].median() if n_closed else None
    median_pnl = closed["pnl_pct"].median() if n_closed else None
    time_stop_rate = (
        closed["position_id"].isin(time_stop_ids).sum() / n_closed if n_closed else None
    )

    rows.append({
        "catalyst_type": ctype,
        "watchlist_entries": len(grp),
        "entered": int(entered),
        "missed": int(missed),
        "missed_rate": round(missed_rate, 3) if missed_rate is not None else None,
        "closed_positions": n_closed,
        "median_days_held": round(median_days, 1) if median_days is not None else None,
        "median_pnl_pct": round(median_pnl, 1) if median_pnl is not None else None,
        "time_stop_rate": round(time_stop_rate, 3) if time_stop_rate is not None else None,
        "enough_data_for_band": total_decided >= MIN_SAMPLES,
        "enough_data_for_hold": n_closed >= MIN_SAMPLES,
    })

summary_df = pd.DataFrame(rows).sort_values("catalyst_type")
summary_df


## 8. Export — this is the ONLY file to share back
Aggregated per-catalyst-type stats only. No symbol names, no prices, no raw
row-level data, no credentials — just counts, rates, and medians.

In [ ]:
from google.colab import files as colab_files

export_path = "/content/stockky_calibration_summary.csv"
summary_df.to_csv(export_path, index=False)
print(f"Saved: {export_path}\n")
print(summary_df.to_string(index=False))

colab_files.download(export_path)


## 9. Next step
Download the CSV this just triggered, and send it back — that's real,
data-backed input for `watchlist_engine/decay.py`'s `CATALYST_PROFILES`/
`EXIT_PROFILES` instead of the original estimates. For any catalyst type
still marked `enough_data_for_band` / `enough_data_for_hold` = False, that
just means keep trading a bit longer and re-run this in a couple of weeks —
that's expected early on, not an error.

In [ ]:
connection.close()
print("Connection closed. Nothing was written, only SELECTed.")
